# Pendulum swing-up — value iteration vs LQR vs PPO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/learn/teaching/pendulum_swing_up_vi_vs_lqr_vs_ppo.ipynb)

This notebook compares three ways to synthesize a policy for the **same** swing-up problem and the **same** quadratic cost $J$. All three return a feedback law $u=\pi(x)$ that tries to minimize $J$.

1. **Value iteration (VI)**: global dynamic programming on a grid — the discretized nonlinear optimum.
2. **LQR**: local linear-quadratic feedback at the upright equilibrium.
3. **PPO**: a neural policy trained by reinforcement learning. The gym reward is $r = -g(x,u,t)\,\Delta t$, so maximizing return is the same as minimizing $J$.

VI is the reference solution of the discretized OCP. LQR is exact only near $\bar x$. PPO is model-free: it never sees $f$, only sampled trajectories.

This page uses the [minilink](https://github.com/alx87grd/minilink) toolbox and [stable-baselines3](https://stable-baselines3.readthedocs.io) for PPO. For the library workflow see [`showcase_minilink`](../intro/showcase_minilink.ipynb); plants, control, and planning are in [`02_dynamics`](../intro/02_dynamics.ipynb), [`03_control`](../intro/03_control.ipynb), and [`09_planning`](../intro/09_planning.ipynb).


In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q gymnasium stable-baselines3")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from minilink.core.diagram import DiagramSystem
from minilink.core.trajectory import Trajectory
from minilink.dynamics.catalog.pendulum.pendulum import InvertedPendulum
from minilink.planning.policy_synthesis import plotting
from minilink.planning.policy_synthesis.discretizer import StateSpaceGrid
from minilink.planning.policy_synthesis.dp import (
    DynamicProgrammingOptions,
    DynamicProgrammingPlanner,
)
from minilink.planning.problems import PlanningProblem
from minilink.interfaces.gymnasium import SB3Controller, Sys2Gym


## 1. Plant

We load a minilink catalog class (`InvertedPendulum`) that already defines the equations of motion and the state/input labels. The state is $x = [\theta,\;\dot\theta]$ and the input is the pivot torque $u$. The dynamics are
$$\dot x = f(x,u).$$
Here $\theta = 0$ is upright, so the hanging-down position is $x_0 = [-\pi,\; 0]$ and the target is $\bar x = [0,\; 0]$. We also set the **domain** — bounds on $x$ and $|u|\le u_{\max}$ — used later by the grid.


In [ ]:
UPRIGHT = np.array([0.0, 0.0])  # target (upright) and LQR linearization point
X0 = np.array([-np.pi, 0.0])  # hanging down
TORQUE = 1.0
DT = 0.05
TF = 10.0
X_GRID = (201, 201)
U_GRID = (21,)
TOL = 0.1
INF = 500.0
Q = np.diag([1.0, 1.0])
R = np.diag([1.0])

In [ ]:
plant = InvertedPendulum()

# Pendulum parameters
plant.params["m"] = 0.1
plant.params["l"] = 0.5
plant.params["I"] = 1.0 / 12.0 * 0.1 * 1.0**2
plant.params["gravity"] = 9.81
plant.params["d"] = 0.0

# Pendulum bounds
plant.state.lower_bound = np.array([-2.0 * np.pi, -12])
plant.state.upper_bound = np.array([+2.0 * np.pi, +12])
plant.inputs["u"].lower_bound = np.array([-TORQUE])
plant.inputs["u"].upper_bound = np.array([+TORQUE])

# Pendulum initial state
plant.x0 = X0.copy()

## 2. Cost function

All three controllers minimize the same quadratic performance metric
$$J = \int_0^{t_f} \big( (x-\bar x)' Q (x-\bar x) + u' R u \big)\, dt.$$
For PPO we will use the equivalent reward $r = -g(x,u)\,\Delta t$, so maximizing return is the same as minimizing $J$.


In [ ]:
from minilink.core.costs import QuadraticCost

cost = QuadraticCost.from_system(plant, xbar=UPRIGHT, Q=Q, R=R)

cost.R[0, 0] = 0.1 / DT
cost.Q[0, 0] = 1.0 / DT
cost.Q[1, 1] = 0.1 / DT

print("Target:", UPRIGHT)
print("Q=\n", cost.Q)
print("R=\n", cost.R)


## 3. Planning problem

A `PlanningProblem` packages the plant, the cost, and the goal. The optimal-control problem is
$$\min_{\pi}\; J \quad\text{s.t.}\quad \dot x = f\big(x,\pi(x)\big),\quad |u|\le u_{\max}.$$
Value iteration solves this on a grid. LQR and PPO target the same $J$ by different approximations.


In [ ]:
problem = PlanningProblem(plant, x_goal=UPRIGHT, cost=cost)


## 4. Value iteration

We discretize $x$ and $u$ on a grid and solve the discrete Bellman equation for the cost-to-go $J^*$:
$$J^*(x) = \min_u \Big\{ g(x,u)\,\Delta t + J^*\big(x + f(x,u)\,\Delta t\big) \Big\}.$$
The minimizing $u$ is the global (discretized) policy $\pi^*(x)$ — the reference solution. Here the state grid is $201\times 201$, the torque has 21 levels, and $\Delta t = 0.05\,\mathrm{s}$.


In [ ]:
grid = StateSpaceGrid(problem, x_grid_shape=X_GRID, u_grid_shape=U_GRID, dt=DT)

planner = DynamicProgrammingPlanner(
    problem,
    grid=grid,
    options=DynamicProgrammingOptions(
        alpha=1.0,
        tol=TOL,
        max_iterations=2000,
        out_of_bound_cost=INF,
        verbose=True,
    ),
)

planner.solve()
planner.clean_infeasible_set()
vi_ctl = planner.get_controller()


In [ ]:
planner.plot_cost2go(jmax=INF)


## 5. LQR

Linearize the plant at the upright equilibrium $(\bar x, \bar u)$:
$$\dot{\tilde x} = A\tilde x + B\tilde u,\qquad \tilde x = x-\bar x.$$
The infinite-horizon LQR gain $K$ minimizes the same quadratic $J$ for this linear model:
$$u = \bar u - K(x-\bar x).$$
Same $Q$, $R$, and plant as value iteration — but no torque limits in the synthesis, and no validity away from $\bar x$.


In [ ]:
from minilink.control.lqr import lqr_at_operating_point

lqr_ctl = lqr_at_operating_point( plant, UPRIGHT, Q, R)
K = lqr_ctl.params["K"]
print("LQR gain K =", np.round(K, 3))


## 6. PPO

PPO is model-free: it never uses $f$ explicitly. We wrap the plant and cost as a gym environment. The reward at each step is
$$r = -g(x,u,t)\,\Delta t,$$
so maximizing expected return is the same as minimizing $J$. Initial states are sampled around the hanging position so the policy must learn swing-up, not only balance.


In [ ]:
env = Sys2Gym(plant, cost, dt=DT, tf=TF)
env.clipping_states = False
env.reset_mode = "uniform"
env.x0_lb = np.array([-np.pi, -6.0])
env.x0_ub = np.array([+np.pi, +6.0])

In [ ]:
#######################################
### NN policy
#######################################

import torch
from stable_baselines3 import PPO


## NN architecture
pi=[256, 256, 256]
vf=[256, 256, 256]

policy_kwargs = dict(activation_fn=torch.nn.ReLU,
                     net_arch=dict(pi=pi, vf=vf))

# device = "cpu"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")


nn = PPO("MlpPolicy", env, verbose=1, policy_kwargs=policy_kwargs, device=device)

In [ ]:
nn.learn(total_timesteps=200000)
print("PPO training done")

ppo_ctl = SB3Controller(nn)


Mote training...

In [ ]:
# nn.learn(total_timesteps=1000000)

## 7. Control laws

The three maps $u=\pi(\theta,\dot\theta)$ on the same state grid. A well-trained PPO policy should resemble VI in the region visited during training. LQR is the linear plane through $\bar x$.


In [ ]:
K_row = lqr_ctl.params["K"][0]
ubar = lqr_ctl.params["ubar"][0]
lqr_law = ubar - (grid.states - UPRIGHT) @ K_row
u_ppo_map, _ = nn.predict(grid.states.astype(np.float32), deterministic=True)

planner.plot_policy()
plotting.plot_value(
    grid, lqr_law, vmin=-TORQUE, vmax=TORQUE, cmap="bwr", title="LQR control law"
)
plotting.plot_value(
    grid, u_ppo_map[:, 0], vmin=-TORQUE, vmax=TORQUE, cmap="bwr", title="PPO policy u[0]"
)


## 8. Closed-loop simulation

We wire each policy as state feedback $u=\pi(x)$ and integrate from the hanging position $x_0 = [-\pi,\; 0]$.


In [ ]:
def closed_loop(controller, x0, name):
    """Wire a state-feedback controller to a fresh copy of the pendulum."""
    plant.x0 = np.array(x0)
    diagram = DiagramSystem()
    diagram.add_subsystem(controller, "ctl")
    diagram.add_subsystem(plant, "plant")
    diagram.connect("plant", "y", "ctl", "x")
    diagram.connect("ctl", "u", "plant", "u")
    diagram.name = name
    diagram.camera_scale = 2.0
    diagram.plot_diagram()
    n_steps = int(TF / DT) + 1  # same step as the DP discretization
    traj = diagram.compute_trajectory(tf=TF, n_steps=n_steps, solver="euler")
    return diagram, plant, traj


def applied_u(controller, traj):
    """Reconstruct u(t) from a controller that implements action(x)."""
    return np.array([controller.action(x) for x in traj.x.T]).T

cl_vi, plant_vi, traj_vi = closed_loop(vi_ctl, X0, "Pendulum with VI")
cl_lqr, plant_lqr, traj_lqr = closed_loop(lqr_ctl, X0, "Pendulum with LQR")
cl_ppo, plant_ppo, traj_ppo = closed_loop(ppo_ctl, X0, "Pendulum with PPO")

cl_vi.plot_trajectory(traj_vi)
cl_lqr.plot_trajectory(traj_lqr)
cl_ppo.plot_trajectory(traj_ppo)


## 9. Animation — VI

Closed-loop motion under the value-iteration policy, from hanging down.


In [ ]:
cl_vi.animate(traj_vi)


## 9. Animation — LQR

Same initial state under LQR. Compare the torque and the path to VI and PPO.


In [ ]:
cl_lqr.animate(traj_lqr)


## 9. Animation — PPO

Same initial state under the trained neural policy.


In [ ]:
cl_ppo.animate(traj_ppo)


## 10. Performance

The same $J$ evaluated along each closed-loop trajectory. VI is the global optimum of the discretized problem. A well-trained PPO policy should approach that $J$; LQR typically spends more torque from the hanging position.


In [ ]:
u_lqr = np.clip(
    ubar - (traj_lqr.x.T - UPRIGHT) @ K_row, -TORQUE, TORQUE
).reshape(1, -1)

traj_vi_cost = cost.evaluate_trajectory(
    Trajectory(t=traj_vi.t, x=traj_vi.x, u=applied_u(vi_ctl, traj_vi))
)
traj_lqr_cost = cost.evaluate_trajectory(
    Trajectory(t=traj_lqr.t, x=traj_lqr.x, u=u_lqr)
)
traj_ppo_cost = cost.evaluate_trajectory(
    Trajectory(t=traj_ppo.t, x=traj_ppo.x, u=applied_u(ppo_ctl, traj_ppo))
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(traj_vi_cost.t, traj_vi_cost.signals["cost"][0], label="VI")
ax.plot(traj_lqr_cost.t, traj_lqr_cost.signals["cost"][0], label="LQR")
ax.plot(traj_ppo_cost.t, traj_ppo_cost.signals["cost"][0], label="PPO")
ax.set_xlabel("t [s]")
ax.set_ylabel("$J$")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print("VI  | J =", round(float(traj_vi_cost.signals["cost"][0, -1]), 1))
print("LQR | J =", round(float(traj_lqr_cost.signals["cost"][0, -1]), 1))
print("PPO | J =", round(float(traj_ppo_cost.signals["cost"][0, -1]), 1))
